In [1]:
import os
import pandas as pd
import numpy as np
# Show all columns
pd.set_option('display.max_columns', None)
import warnings

warnings.filterwarnings("ignore")

In [2]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


## Download image data using provided download CLI
Below we download all metadata and resized images (500p) for a FungiTastic-Mini subset.

In [3]:
# !python ../../dataset/download.py --metadata --images --subset "m" --size "500" --save_path "./"

In [4]:
!ls ./FungiTastic/metadata/FungiTastic-Mini/

FungiTastic-Mini-ClosedSet-Test.csv FungiTastic-Mini-OpenSet-Test.csv
FungiTastic-Mini-ClosedSet-Val.csv  FungiTastic-Mini-OpenSet-Val.csv
FungiTastic-Mini-DNA-Test.csv       FungiTastic-Mini-Train.csv


In [5]:
!ls ./FungiTastic/FungiTastic-Mini

dna-test test     train    val


## Load and preprocess metadata

In [6]:
print(os.getcwd())

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/baselines/closed_set


In [7]:
BASE_METADATA_PATH = f"{os.getcwd()}/FungiTastic/metadata/FungiTastic-Mini/"

train_df = pd.read_csv(f"{BASE_METADATA_PATH}FungiTastic-Mini-Train.csv")
val_df = pd.read_csv(f"{BASE_METADATA_PATH}FungiTastic-Mini-ClosedSet-Val.csv")

### Setting image_path to the images.

In [8]:
BASE_IMAGE_PATH = "./FungiTastic/FungiTastic-Mini"
SIZE = "500p"

train_df["image_path"] = train_df.filename.apply(lambda filename: f"{BASE_IMAGE_PATH}/train/{SIZE}/{filename}")
val_df["image_path"] = val_df.filename.apply(lambda filename: f"{BASE_IMAGE_PATH}/val/{SIZE}/{filename}")

In [9]:
print("Shape of train_df: ", train_df.shape)
print("Shape of val_df: ", val_df.shape)

Shape of train_df:  (46842, 32)
Shape of val_df:  (9412, 32)


## Remove Taxonomic Features

In [10]:
train_df = train_df[['species', 'year', 'month', 'day', 'habitat', 'countryCode','hasCoordinate','iucnRedListCategory','substrate','latitude','longitude','coorUncert','region','district','metaSubstrate','poisonous','elevation','landcover','biogeographicalRegion', 'image_path']]

In [11]:
val_df = val_df[['species', 'year', 'month', 'day', 'habitat', 'countryCode','hasCoordinate','iucnRedListCategory','substrate','latitude','longitude','coorUncert','region','district','metaSubstrate','poisonous','elevation','landcover','biogeographicalRegion', 'image_path']]

## Analyze number of nan rows and drop them

In [12]:
train_df_num_nan_rows = train_df.isna().any(axis=1).sum()
print("train_df number of nan rows: ", train_df_num_nan_rows)

val_df_num_nan_rows = val_df.isna().any(axis=1).sum()
print("val_df number of nan rows: ", val_df_num_nan_rows)

print("Percentage nan rows train_df: ", train_df_num_nan_rows / train_df.shape[0])
print("Percentage nan rows val_df: ", val_df_num_nan_rows / val_df.shape[0])

train_df number of nan rows:  2414
val_df number of nan rows:  21
Percentage nan rows train_df:  0.051534947269544426
Percentage nan rows val_df:  0.0022311942201444965


In [13]:
train_df.dropna(inplace=True)
val_df.dropna(inplace=True)

In [14]:
print("Shape of train_df after dropping nan rows: ", train_df.shape)
print("Shape of val_df after dropping nan rows: ", val_df.shape)

Shape of train_df after dropping nan rows:  (44428, 20)
Shape of val_df after dropping nan rows:  (9391, 20)


In [15]:
train_df.columns

Index(['species', 'year', 'month', 'day', 'habitat', 'countryCode',
       'hasCoordinate', 'iucnRedListCategory', 'substrate', 'latitude',
       'longitude', 'coorUncert', 'region', 'district', 'metaSubstrate',
       'poisonous', 'elevation', 'landcover', 'biogeographicalRegion',
       'image_path'],
      dtype='object')

## Analyze distribution of labels and narrow to most frequent N

In [16]:
train_species_count = train_df['species'].value_counts()

train_proportions = train_df['species'].value_counts(normalize=True)
summary = pd.DataFrame({
    'Count': train_species_count,
    'Proportion': train_proportions,
})
summary.describe()

,Count,Proportion
count,210.000000,210.000000
mean,211.561905,0.004762
std,266.743302,0.006004
min,5.000000,0.000113
25%,50.250000,0.001131
50%,105.500000,0.002375
75%,267.750000,0.006027
max,1745.000000,0.039277


In [17]:
print("The largest frequency of any species: ", max(train_species_count.values))
print("The smallest frequency of any species: ",min(train_species_count.values))

The largest frequency of any species:  1745
The smallest frequency of any species:  5


In [18]:
train_df_count_quantiles = train_df["species"].value_counts().quantile([0.25, 0.5, 0.75])

print("Frequency at quantiles: ", "\n", train_df_count_quantiles)

Frequency at quantiles:  
 0.25     50.25
0.50    105.50
0.75    267.75
Name: count, dtype: float64


In [20]:
min_samples = 350  # or whatever makes sense for your model
valid_classes = (train_species_count[train_species_count >= min_samples])
N = len(valid_classes)
print("Number of classes with at least 150 examples: ", N)
print("Number of total classes: ", len(train_species_count))
print("Percent of total classes with at least 150 examples: ", N / len(train_species_count))

Number of classes with at least 150 examples:  38
Number of total classes:  210
Percent of total classes with at least 150 examples:  0.18095238095238095


In [21]:
sum_of_min_sample_classes = sum(list(dict(train_species_count).values())[:N])
print("Total records of classes with at least 150 samples: ", sum_of_min_sample_classes)

pct_w_min_samples = sum_of_min_sample_classes / train_df.shape[0]
print("Percent of examples of classes with minimum number of examples: ", f"{pct_w_min_samples * 100:.2f}%")

Total records of classes with at least 150 samples:  24992
Percent of examples of classes with minimum number of examples:  56.25%


In [22]:
min_150_labels = valid_classes.keys()

In [23]:
train_df = train_df[train_df['species'].isin(min_150_labels)]
val_df = val_df[val_df['species'].isin(min_150_labels)]

print("Shape of train_df: ", train_df.shape)
print("Shape of val_df: ", val_df.shape)

Shape of train_df:  (24992, 20)
Shape of val_df:  (6062, 20)


## Split Train into Train and Test
- Because the test df that came from FungiTastic doesn't have labels, we will split the train into train and test.

In [24]:
from sklearn.model_selection import train_test_split

In [25]:
X = train_df[list(set(train_df.columns) - set(['species']))]
Y = train_df['species']

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.30, random_state=1234, stratify=Y
)

X_val = val_df[list(set(val_df.columns) - set(['species']))]
Y_val = val_df['species']

In [26]:
print("Shape of X_train: ", X_train.shape)
print("Shape of Y_train: ", Y_train.shape)

print("Shape of X_test: ", X_test.shape)
print("Shape of Y_test: ", Y_test.shape)

print("Shape of X_val: ", X_val.shape)
print("Shape of Y_val: ", Y_val.shape)

Shape of X_train:  (17494, 19)
Shape of Y_train:  (17494,)
Shape of X_test:  (7498, 19)
Shape of Y_test:  (7498,)
Shape of X_val:  (6062, 19)
Shape of Y_val:  (6062,)


## Preprocess Categorical Metadata

In [56]:
metadata_feat_list = [
    'year', 'month', 'day', 'habitat', 'countryCode',
       'hasCoordinate', 'iucnRedListCategory', 'substrate', 'latitude',
       'longitude', 'coorUncert', 'region', 'district', 'metaSubstrate',
       'poisonous', 'elevation', 'landcover', 'biogeographicalRegion'
]
X_train_metadata = X_train[metadata_feat_list]
X_val_metadata = X_val[metadata_feat_list]
X_test_metadata = X_test[metadata_feat_list]

### Frequency Encoding

In [57]:
cat_metadata_feat_list = [
    'habitat', 'countryCode', 'hasCoordinate',
    'iucnRedListCategory', 'substrate', 'coorUncert', 'region', 'district', 
    'metaSubstrate', 'biogeographicalRegion'
]

X_train_cat_metadata = X_train_metadata[cat_metadata_feat_list]
X_val_cat_metadata = X_val_metadata[cat_metadata_feat_list]
X_test_cat_metadata = X_test_metadata[cat_metadata_feat_list]

In [58]:
train_cat_vc_dict = {}

for cat in list(X_train_cat_metadata.columns):
    train_cat_vc_dict[cat] = dict(X_train_cat_metadata[cat].value_counts())

In [59]:
X_train_cat_metadata_enc = X_train_cat_metadata.copy()
for col, mapping in train_cat_vc_dict.items():
    # Use .map with dict (fast, vectorized, memory-friendly)
    X_train_cat_metadata_enc[col] = X_train_cat_metadata[col].map(mapping)

In [60]:
X_val_cat_metadata_enc = X_val_cat_metadata.copy()
for col, mapping in train_cat_vc_dict.items():
    # Use .map with dict (fast, vectorized, memory-friendly)
    X_val_cat_metadata_enc[col] = X_val_cat_metadata[col].map(mapping)

In [61]:
X_test_cat_metadata_enc = X_test_cat_metadata.copy()
for col, mapping in train_cat_vc_dict.items():
    # Use .map with dict (fast, vectorized, memory-friendly)
    X_test_cat_metadata_enc[col] = X_test_cat_metadata[col].map(mapping)

### Coordinates Parsing

In [62]:
X_train_cat_metadata_enc["lat_sin"] = np.sin(np.radians(X_train["latitude"]))
X_train_cat_metadata_enc["lat_cos"] = np.cos(np.radians(X_train["latitude"]))
X_train_cat_metadata_enc["lon_sin"] = np.sin(np.radians(X_train["longitude"]))
X_train_cat_metadata_enc["lon_cos"] = np.cos(np.radians(X_train["longitude"]))

In [63]:
X_val_cat_metadata_enc["lat_sin"] = np.sin(np.radians(X_val["latitude"]))
X_val_cat_metadata_enc["lat_cos"] = np.cos(np.radians(X_val["latitude"]))
X_val_cat_metadata_enc["lon_sin"] = np.sin(np.radians(X_val["longitude"]))
X_val_cat_metadata_enc["lon_cos"] = np.cos(np.radians(X_val["longitude"]))

In [64]:
X_test_cat_metadata_enc["lat_sin"] = np.sin(np.radians(X_test["latitude"]))
X_test_cat_metadata_enc["lat_cos"] = np.cos(np.radians(X_test["latitude"]))
X_test_cat_metadata_enc["lon_sin"] = np.sin(np.radians(X_test["longitude"]))
X_test_cat_metadata_enc["lon_cos"] = np.cos(np.radians(X_test["longitude"]))

In [65]:
####

#### Standardize Numerical Features

In [66]:
numerical_feats = ['elevation', 'landcover']

X_train_numerical = X_train[numerical_feats]
X_val_numerical = X_val[numerical_feats]
X_test_numerical = X_test[numerical_feats]

In [67]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()

X_train_numerical_std = std_scaler.fit_transform(X_train_numerical)
X_val_numerical_std = std_scaler.transform(X_val_numerical)
X_test_numerical_std = std_scaler.transform(X_test_numerical)

### Combine Metadata encoded with Numerical fields

In [68]:
non_numerical_nor_cat_feats = ['year', 'month', 'day', 'poisonous']

X_train_non_numerical_nor_cat = np.array(X_train[non_numerical_nor_cat_feats])
X_train_cat_metadata_enc = np.array(X_train_cat_metadata_enc)
X_train_img_paths = np.array(X_train['image_path'].copy()).reshape(-1)

X_train_prcsd = np.concatenate([X_train_cat_metadata_enc, X_train_non_numerical_nor_cat, X_train_numerical_std], axis=1)

In [69]:
X_val_non_numerical_nor_cat = np.array(X_val[non_numerical_nor_cat_feats])
X_val_cat_metadata_enc = np.array(X_val_cat_metadata_enc)
X_val_img_paths = np.array(X_val['image_path'].copy()).reshape(-1)

X_val_prcsd = np.concatenate([X_val_cat_metadata_enc, X_val_non_numerical_nor_cat, X_val_numerical_std], axis=1)

In [70]:
X_test_non_numerical_nor_cat = np.array(X_test[non_numerical_nor_cat_feats])
X_test_cat_metadata_enc = np.array(X_test_cat_metadata_enc)
X_test_img_paths = np.array(X_test['image_path'].copy()).reshape(-1)

X_test_prcsd = np.concatenate([X_test_cat_metadata_enc, X_test_non_numerical_nor_cat, X_test_numerical_std], axis=1)

### Preprocess Images

In [71]:
import tensorflow as tf

In [72]:
print(os.getcwd())

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/baselines/closed_set


In [73]:
def load_and_preprocess_image(path):
    path = tf.squeeze(path)
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [224, 224])
    image = image / 255.0
    return image

def maybe_augment_image(image, augment_prob=0.1):
    # Draw a random number in [0,1)
    rand_val = tf.random.uniform([], 0, 1)
    
    def augment_fn():
        # Apply brightness or other augmentations here
        image_aug = tf.image.random_brightness(image, max_delta=0.2)
        return tf.clip_by_value(image_aug, 0.0, 1.0)
    
    # Apply augmentation with given probability
    return tf.cond(rand_val < augment_prob, augment_fn, lambda: image)

def process_row(sample, label):
    image = load_and_preprocess_image(sample['image'])
    image = maybe_augment_image(image, augment_prob=0.1)
    return ({"image": image, "metadata": sample['metadata']}, label)

In [74]:
# def process_row(sample, label):
#     image = load_and_preprocess_image(sample['image'])
#     return ({"image": image, "metadata": sample['metadata']}, label)

In [75]:
def standardize_image(data, label, mean, std):
    image = (tf.cast(data['image'], tf.float32) - mean) / std
    return {"image": image, "metadata": data['metadata']}, label

### Train set image processing

In [76]:
from PIL import Image

#### 1. Take random 10% of training image paths

In [47]:
sample_size = int(0.1 * len(X_train_img_paths))
aug_indices = np.random.choice(len(X_train_img_paths), sample_size, replace=False)

In [48]:
aug_paths = X_train_img_paths[aug_indices].copy()

In [49]:
def read_image_as_numpy_array(path):
    """
    Reads an image from path, applies brightness adjustment, and returns a NumPy array.
    delta > 0 => brighter; delta < 0 => darker.
    """
    # Read and preprocess
    img = Image.open(path).convert("RGB").resize((224, 224))
    img = np.array(img, dtype=np.float32) / 255.0  # normalize to [0,1]

    # Convert to TensorFlow tensor
    tf_img = tf.convert_to_tensor(img, dtype=tf.float32)

    # Clip to valid range and convert back to NumPy
    tf_img = tf.clip_by_value(tf_img, 0.0, 1.0)
    return tf_img.numpy()

In [50]:
def read_image_as_numpy_array_augment_brightness(path, delta=0.2):
    """
    Reads an image from path, applies brightness adjustment, and returns a NumPy array.
    delta > 0 => brighter; delta < 0 => darker.
    """
    # Read and preprocess
    img = Image.open(path).convert("RGB").resize((224, 224))
    img = np.array(img, dtype=np.float32) / 255.0  # normalize to [0,1]

    # Convert to TensorFlow tensor
    tf_img = tf.convert_to_tensor(img, dtype=tf.float32)

    # Random brightness adjustment
    delta_val = tf.random.uniform([], -delta, delta)  # random brightness change
    bright_img = tf.image.adjust_brightness(tf_img, delta_val)

    # Clip to valid range and convert back to NumPy
    bright_img = tf.clip_by_value(bright_img, 0.0, 1.0)
    return bright_img.numpy()

In [63]:
augmented_images = np.array([read_image_as_numpy_array_augment_brightness(p) for p in aug_paths])

In [ ]:
X_train_images = np.array([read_image_as_numpy_array(p) for p in X_train_img_paths])

#### Copy the corresponding metadata rows and append to end of metadata array

In [52]:
aug_X_train_prcsd = X_train_prcsd[aug_indices].copy()
X_train_prcsd = np.concatenate([X_train_prcsd, aug_X_train_prcsd])

## Current Iteration

#### Target Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical

le = LabelEncoder()
Y_train_int = le.fit_transform(Y_train)          # convert strings → integers
Y_train_onehot = to_categorical(Y_train_int)     # convert integers → one-hot

In [86]:
Y_val_int = le.transform(Y_val)          # convert strings → integers
Y_val_onehot = to_categorical(Y_val_int)  

#### Train TF Dataset and standardization

In [81]:
# Build dataset from (image_paths, metadata_features, labels)
train_dataset = tf.data.Dataset.from_tensor_slices(({
    "image": X_train_img_paths,
    "metadata": X_train_prcsd
}, Y_train_onehot))
train_dataset = train_dataset.map(process_row, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(1000).batch(32)

In [82]:
# Initialize accumulators
n_pixels = 0
channel_sum = np.zeros(3, dtype=np.float64)
channel_sum_sq = np.zeros(3, dtype=np.float64)

for features, _ in train_dataset:  # features = {"image": ..., "metadata": ...}
    images = features["image"].numpy()  # shape: (batch, 224, 224, 3)
    
    # Flatten spatial dimensions
    pixels = images.reshape(-1, 3)   # (batch * 224 * 224, 3)
    
    # Accumulate sums
    channel_sum += pixels.sum(axis=0)
    channel_sum_sq += np.square(pixels).sum(axis=0)
    n_pixels += pixels.shape[0]

# Compute mean and std
channel_mean = channel_sum / n_pixels
channel_std = np.sqrt(channel_sum_sq / n_pixels - np.square(channel_mean))

print("Channel Mean:", channel_mean)
print("Channel Std:", channel_std)

Channel Mean: [0.41768076 0.39992564 0.329903  ]
Channel Std: [0.25829314 0.2476494  0.25187153]


2025-11-10 18:55:09.581773: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [83]:
mean = tf.constant(channel_mean, dtype=tf.float32)
std = tf.constant(channel_std, dtype=tf.float32)

In [84]:
standardized_train_dataset = train_dataset.map(
    lambda data, label: standardize_image(data, label, mean, std), 
    num_parallel_calls=tf.data.AUTOTUNE
)

### Validation set image processing

In [87]:
# Build dataset from (image_paths, metadata_features, labels)
val_dataset = tf.data.Dataset.from_tensor_slices(({
    "image": X_val_img_paths,
    "metadata": X_val_prcsd
}, Y_val_onehot))

val_dataset = val_dataset.map(process_row, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.shuffle(1000).prefetch(tf.data.AUTOTUNE)

In [88]:
standardized_val_dataset = val_dataset.map(
    lambda data, label: standardize_image(data, label, mean, std), 
    num_parallel_calls=tf.data.AUTOTUNE
)

## Build, Compile, and Fit Model taking Image Tensors and Metadata Tensors

In [ ]:
from keras import layers, models, Input

# Image branch
image_input = Input(shape=(224, 224, 3), name='image')
x = layers.Conv2D(48, (3, 3), activation='relu')(image_input)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, (3, 3), activation='relu')(x)
x = layers.MaxPooling2D()(x)
x = layers.Flatten()(x)

# Metadata branch
metadata_input = Input(shape=(20,), name='metadata')
m = layers.Dense(128, activation='relu')(metadata_input)
m = layers.Dense(64, activation='relu')(metadata_input)(m)

# Combine branches
combined = layers.concatenate([x, m])
z = layers.Dense(128, activation='relu')(combined)
z = layers.Dense(64, activation='relu')(combined)(z)

# Final output softmax layer
output = layers.Dense(len(le.classes_), activation='softmax')(z)

# Build model
model = models.Model(inputs={"image": x, "metadata": m}, outputs=output)

In [93]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 222, 222,  │      1,344 │ image[0][0]       │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 111, 111,  │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 109, 109,  │     27,712 │ max_pooling2d_2[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 54, 54,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ metadata            │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 186624)    │          0 │ max_pooling2d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      1,344 │ metadata[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 186688)    │          0 │ flatten_1[0][0],  │
│ (Concatenate)       │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │ 23,896,192 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 89)        │     11,481 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,938,073 (91.32 MB)

 Trainable params: 23,938,073 (91.32 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [96]:
history = model.fit(
    standardized_train_dataset,
    epochs=10,
    # validation_data=val_dataset_standardized,  # if you have a validation dataset
    batch_size=64
)

Epoch 1/10


809/809 ━━━━━━━━━━━━━━━━━━━━ 3165s 4s/step - accuracy: 0.0346 - loss: 9.9210
Epoch 2/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 25169s 31s/step - accuracy: 0.0387 - loss: 4.3538
Epoch 3/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 17956s 22s/step - accuracy: 0.0472 - loss: 4.3089
Epoch 4/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 614s 758ms/step - accuracy: 0.0472 - loss: 4.2898
Epoch 5/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 2331s 3s/step - accuracy: 0.0472 - loss: 4.2830
Epoch 6/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 805s 995ms/step - accuracy: 0.0472 - loss: 4.2806
Epoch 7/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 6077s 8s/step - accuracy: 0.0472 - loss: 4.2797
Epoch 8/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 640s 789ms/step - accuracy: 0.0472 - loss: 4.2794
Epoch 9/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 4253s 5s/step - accuracy: 0.0472 - loss: 4.2793
Epoch 10/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 526s 649ms/step - accuracy: 0.0472 - loss: 4.2792


In [99]:
history.history['accuracy'][-1]

0.04719752445816994

In [103]:
image_input = Input(shape=(224, 224, 3), name='image')
x = layers.Conv2D(48, (3, 3), activation='relu')(image_input)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, (3, 3), activation='relu')(x)
x = layers.MaxPooling2D()(x)
x = layers.Flatten()(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dense(64, activation='relu')(x)

# Metadata branch
metadata_input = Input(shape=(20,), name='metadata')
m = layers.Dense(128, activation='relu')(metadata_input)
m = layers.Dense(64, activation='relu')(m)

# Combine branches
combined = layers.concatenate([x, m])
z = layers.Dense(128, activation='relu')(combined)
z = layers.Dense(64, activation='relu')(z)

# Final output softmax layer
output = layers.Dense(len(le.classes_), activation='softmax')(z)

# Build model
model2 = models.Model(inputs={"image": image_input, "metadata": metadata_input}, outputs=output)

In [104]:
model2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model2.fit(
    standardized_train_dataset,
    epochs=10,
    # validation_data=val_dataset_standardized,  # if you have a validation dataset
    batch_size=128
)

Epoch 1/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 617s 759ms/step - accuracy: 0.0407 - loss: 37.2992
Epoch 2/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 699s 862ms/step - accuracy: 0.0472 - loss: 4.3466
Epoch 3/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 781s 963ms/step - accuracy: 0.0472 - loss: 4.3031
Epoch 4/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 750s 925ms/step - accuracy: 0.0472 - loss: 4.2876
Epoch 5/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 694s 856ms/step - accuracy: 0.0471 - loss: 4.4208
Epoch 6/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 1165s 1s/step - accuracy: 0.0472 - loss: 4.2804
Epoch 7/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 1365s 2s/step - accuracy: 0.0472 - loss: 4.2797
Epoch 8/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 1047s 1s/step - accuracy: 0.0472 - loss: 4.2794
Epoch 9/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 819s 1s/step - accuracy: 0.0472 - loss: 4.2792
Epoch 10/10
809/809 ━━━━━━━━━━━━━━━━━━━━ 762s 940ms/step - accuracy: 0.0472 - loss: 4.2792
